# Discover aspect vocabulary with sentence windows

Test on the first `N_GAMES` cached games, ordered by app ID. Each sentence anchors a window: previous + current + next (`RADIUS=1`), clipped at review boundaries. An aspect matches only when a seed occurs in the **central sentence**. Set `RADIUS=0` to compare sentence-only contexts.

Count candidate presence once per window. **Lift = fraction of aspect windows containing the candidate / fraction of all windows containing it.** Both use identical window boundaries. Overlapping windows are correlated; distinct-review and distinct-game support help filter repeated evidence. Blank reviews and duplicate review IDs per game are skipped.

Uses the existing parser and vocabulary. Candidates include single words and adjacent bigrams within sentences. Exclude known aspect seeds and candidates containing stop words, negations, numbers, or words shorter than three characters. Aspect seeds may also be bigrams. Associations are suggestions, not confirmed aspect terms: inspect examples before adding words. No files or vocabularies are changed; no extra dependencies. Run from the project root or notebooks/.

In [1]:
import sys
from collections import Counter, defaultdict
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.csv_io import REVIEWS_DATA_DIR, load_reviews
from src.review_parser import ReviewParser
from src.semantic_analyzer import SemanticAnalyzer

N_GAMES, RADIUS = 1000, 1
MIN_REVIEWS, MIN_GAMES, MIN_LIFT = 100, 50, 2
TOP_N = 15
parser = ReviewParser()
aspects = SemanticAnalyzer().aspects
excluded = parser.stop_words | parser.NEGATION_WORDS | set.union(*aspects.values())
files = sorted((p for p in REVIEWS_DATA_DIR.glob("*.csv") if p.stem.isdigit()),
               key=lambda p: int(p.stem))[:N_GAMES]


In [2]:
background, aspect_windows = Counter(), Counter()
cooccurrence = {a: Counter() for a in aspects}
review_support = {a: Counter() for a in aspects}
game_support = {a: Counter() for a in aspects}
seeds, examples = defaultdict(set), {}
n_windows = n_reviews = 0

for path in files:
    game_hits = {a: set() for a in aspects}
    reviews = {r.recommendationid: r for r in load_reviews(int(path.stem)) if r.review.strip()}
    for review in reviews.values():
        n_reviews += 1
        sentences = parser.split_to_sentences(review.review)
        tokens = [set(words) | {a + " " + b for a, b in zip(words, words[1:])}
                  for words in parser.tokenize(sentences)]
        review_hits = {a: set() for a in aspects}
        for i, center in enumerate(tokens):
            start, end = max(0, i - RADIUS), i + RADIUS + 1
            window = set().union(*tokens[start:end])
            words = {w for w in window if w not in excluded
                     and all(part.isalpha() and len(part) >= 3 and part not in excluded
                             for part in w.split())}
            background.update(words)
            n_windows += 1
            for aspect, vocabulary in aspects.items():
                matched = center & vocabulary
                if not matched:
                    continue
                aspect_windows[aspect] += 1
                cooccurrence[aspect].update(words)
                review_hits[aspect].update(words)
                for word in words:
                    seeds[aspect, word].update(matched)
                    if (aspect, word) not in examples:
                        examples[aspect, word] = (review.appid, review.recommendationid,
                                                 " ".join(sentences[start:end])[:600])
        for aspect in aspects:
            review_support[aspect].update(review_hits[aspect])
            game_hits[aspect].update(review_hits[aspect])
    for aspect in aspects:
        game_support[aspect].update(game_hits[aspect])

print(f"{len(files)} games, {n_reviews:,} reviews, {n_windows:,} windows")


969 games, 92,488 reviews, 1,380,677 windows


In [3]:
candidates = {}
for aspect in aspects:
    ranked = []
    for word, count in cooccurrence[aspect].items():
        lift = (count / aspect_windows[aspect]) / (background[word] / n_windows)
        if (review_support[aspect][word] >= MIN_REVIEWS
                and game_support[aspect][word] >= MIN_GAMES and lift >= MIN_LIFT):
            ranked.append({"word": word, "lift": lift, "windows": count,
                           "reviews": review_support[aspect][word],
                           "games": game_support[aspect][word],
                           "seeds": sorted(seeds[aspect, word])})
    candidates[aspect] = sorted(ranked, key=lambda r: (-r["lift"], -r["reviews"], r["word"]))
    print(f"\n{aspect.upper()} — {aspect_windows[aspect]:,} matching windows")
    print(f"{'word':20} {'lift':>6} {'reviews':>8} {'games':>6}  matched seeds")
    for row in candidates[aspect][:TOP_N]:
        print(f"{row['word']:20} {row['lift']:6.2f} {row['reviews']:8} {row['games']:6}  {', '.join(row['seeds'])}")



COMBAT — 34,789 matching windows
word                   lift  reviews  games  matched seeds
ranged                13.70      423    200  blocking, combat, combos, dodging, fighting, melee, parry, parrying, weapons
bows                  11.97      112     65  blocking, combat, combos, dodging, duels, fighting, melee, parry, parrying, weapons
pistols               10.02      144    101  blocking, combat, combos, dodging, fighting, melee, weapons
each weapon            9.55      106     73  blocking, combat, combos, fighting, melee, weapons
shotguns               9.24      147     93  combat, combos, dodging, fighting, melee, weapons
armors                 9.03      110     79  blocking, combat, combos, fighting, melee, parry, weapons
arsenal                8.38      248    162  blocking, combat, combos, dodging, duels, fighting, melee, parry, weapons
fluid                  8.12      410    195  blocking, brawling, combat, combos, dodging, duels, fighting, melee, parry, parrying, weapons